Import simulation outputs from LISFLOOD-FP and package as NetCDF files.

Ensure to specify `voutput` in LISFLOOD-FP parameters file to get velocity as well as water depth.

In [ ]:
import numpy as np
import os
import rioxarray as rxr
from typing import cast, Literal
import xarray as xr

import graph_creation

# LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/res_5m_training_data"
# DEM_FILE = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/DEM/DEM_0.xyz"
# POLYGON_FILE = "raw_datasets_dyce/Geometry/polygon_0.pol"
# PREFIX = "res_5m_acc_cuda"
# MAX_STEP = 400

LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/res_dk15_hydrograph102"
DEM_FILE = "/home/aidan/code/data/res_dk15_hydrograph102/res_dk15_hydrograph102.dem"
POLYGON_FILE = "raw_datasets_dyce/Geometry/polygon_0.pol"
PREFIX = "res_dk15_hydrograph102"
MAX_STEP = 97


def read_step(step: int, prefix: str, ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"]|None=None) -> xr.DataArray:
        return cast(xr.DataArray, rxr.open_rasterio(os.path.join(LISFLOOD_OUTPUT_DIR, f"{prefix}-{int(step):04}.{ftype}"), parse_coordinates=True, masked=True))[0]

/home/aidan/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

POLYGON_GEOJSON_PATH = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/Geometry/dyce_poly_10m.geojson"
with open(POLYGON_GEOJSON_PATH) as f:
    polygon_json = json.load(f)
polygon = np.array(polygon_json["features"][0]["geometry"]["coordinates"][0][0])
min, max = np.min(polygon, axis=0), np.max(polygon, axis=0)

coords = "\n".join([", ".join(line.astype(str)) for line in polygon])
with open("polygon_0.pol", "w") as f:
    f.write(f"# Extent: ({min[0]}, {min[1]}, {max[0]}, {max[1]})\n# Coordinates (x, y):\n")
    f.write(coords)

In [2]:
def extract_parameter(ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"], max_step, prefix=PREFIX, shape=None):
    results = []
    for step in range(0, max_step+1):
        results.append(read_step(step, prefix, ftype))
    parameter_array = xr.concat(results, "time")
    
    return parameter_array

In [ ]:
from meshkernel import GeometryList, MeshKernel, ProjectToLandBoundaryOption, OrthogonalizationParameters, MeshRefinementParameters
from graph_creation import Mesh
with open(POLYGON_FILE) as file:
    boundary_nodes = np.array([[value for value in line.strip().split(",")] for line in file.readlines()[2:]], dtype=np.double)

boundary_polygon = GeometryList(boundary_nodes[:,0].copy(), boundary_nodes[:,1].copy())
print(boundary_nodes.shape)
meshes = []

mk = MeshKernel()
# mk.mesh2d_make_triangular_mesh_from_polygon(boundary_polygon)
mk.mesh2d_make_triangular_mesh_from_samples(boundary_polygon)



print(mk.mesh2d_get().edge_faces)
print(mk.mesh2d_get_orthogonality())

# print(mk.mesh2d_get_mesh_boundaries_as_polygons().x_coordinates)
# print(mk.mesh2d_get_mesh_boundaries_as_polygons().y_coordinates)
refinement_parameters = MeshRefinementParameters(refine_intersected=True, min_edge_size=0.5, 
                                                max_refinement_iterations=1, smoothing_iterations=5)
mk.mesh2d_refine_based_on_polygon(boundary_polygon, refinement_parameters)

mk.mesh2d_compute_orthogonalization(ProjectToLandBoundaryOption(0), OrthogonalizationParameters(
                    outer_iterations=25, boundary_iterations=25, inner_iterations=25, 
                    orthogonalization_to_smoothing_factor=0.975),
                    boundary_polygon, inner_boundary_polygon)
# print(mk.mesh2d_get_mesh_boundaries_as_polygons().x_coordinates)
# print(mk.mesh2d_get_mesh_boundaries_as_polygons().y_coordinates)

mesh = Mesh()
mesh._import_from_meshkernel(mk)
meshes.append(mesh)



# mk.mesh2d_compute_orthogonalization(ProjectToLandBoundaryOption(0), OrthogonalizationParameters(
#                     outer_iterations=25, boundary_iterations=1, inner_iterations=25, 
#                     orthogonalization_to_smoothing_factor=0.975),
#                     boundary_polygon, inner_boundary_polygon)
# mesh = Mesh()
# mesh._import_from_meshkernel(mk)
# meshes.append(mesh)

# refinement_parameters = MeshRefinementParameters(refine_intersected=True, min_edge_size=0.5, 
#                                                 max_refinement_iterations=1, smoothing_iterations=5)
# mk.mesh2d_refine_based_on_polygon(boundary_polygon, refinement_parameters)

# mk.mesh2d_compute_orthogonalization(ProjectToLandBoundaryOption(0), OrthogonalizationParameters(
#                     outer_iterations=25, boundary_iterations=25, inner_iterations=25, 
#                     orthogonalization_to_smoothing_factor=0.975),
#                     boundary_polygon, inner_boundary_polygon)
# mesh = Mesh()
# mesh._import_from_meshkernel(mk)
# meshes.append(mesh)

(11, 2)
[ 2  3  2 -1  1  2  3 -1  3  4  0  1  1 -1  0 -1  0]


In [8]:
plt.plot(mk.mesh2d_get_mesh_boundaries_as_polygons().x_coordinates, mk.mesh2d_get_mesh_boundaries_as_polygons().y_coordinates, color="blue")
plt.plot(boundary_nodes[:,0], boundary_nodes[:,1], color="red")

NameError: name 'plt' is not defined

In [5]:
pointsource_x, pointsource_y = 388911, 814101 # The true pointsource coordinates

In [13]:
import matplotlib.pyplot as plt
%matplotlib qt
fig, ax = plt.subplots()
meshes[-1].meshtmp.plot_faces(ax)
ax.scatter(pointsource_x, pointsource_y, color="red")
plt.show()

NameError: name 'pointsource_x' is not defined

In [15]:
import importlib
graph_creation = importlib.reload(graph_creation)

In [3]:
meshes = graph_creation.create_mesh_dhydro("raw_datasets_dyce/Geometry/polygon_0.pol", 5, False)
# meshes = graph_creation.create_mesh_dhydro("raw_datasets_dk15/Geometry/polygon_102.pol", 1, False)
mesh = meshes[-1] # Highest resolution mesh

In [4]:
for mesh in meshes:
    mesh._import_DEM(DEM_FILE)

In [5]:
mesh

Mesh object with 21881 nodes, 65111 edges, 43231 faces, and 129230 dual edges

In [6]:
boundary_edge_mask = mesh.edge_type == 1 # Mask out non-boundary edges (edges of type 1)

# Find point source node for input boundary conditions
pointsource_x, pointsource_y = 388911, 814101 # The true pointsource coordinates
# pointsource_x, pointsource_y = 389000, 813950 # Start of the river on the smooth-edged DEM
edge_xy = (mesh.node_xy[mesh.edge_index[1]] + mesh.node_xy[mesh.edge_index[0]]) / 2
dist_to_pointsource = np.ma.array(np.abs(edge_xy[:,0]-pointsource_x) + np.abs(edge_xy[:,1]-pointsource_y), mask=boundary_edge_mask)
bc_edge_index = np.argmin(dist_to_pointsource)
print(edge_xy[bc_edge_index])
print("Distance: ", np.sqrt(np.sum((edge_xy[bc_edge_index] - np.array([pointsource_x, pointsource_y]))**2)) )

# Set edge type to BC edge
mesh.edge_type[bc_edge_index] = 2

# Correctly format edge_faces
edge_faces = mesh.edge_faces.reshape(-1,2)
if edge_faces[bc_edge_index][0] != -1:
    # If the first -1 (indicating no face on this side) isn't first
    edge_faces[bc_edge_index] = edge_faces[bc_edge_index, ::-1] # Swap the node indices of the edges (graph_creation.Mesh._import_from_map_netcdf relies on this to identify the BC edge)


[388930.64821206 814102.47557284]
Distance:  19.70354161490951


In [20]:
edge_faces[3172]

array([  -1, 2596], dtype=int32)

In [354]:
mesh.edge_faces.shape

(3702,)

In [323]:
edge_faces.shape

(221604, 2)

In [348]:
mesh.edge_type.shape

(4201,)

In [7]:
face_coords = xr.Dataset(
    coords={
        "mesh2d_nFaces": ("mesh2d_nFaces", range(mesh.face_xy.shape[0])),
        "x": ("mesh2d_nFaces", mesh.face_xy[:, 0]),
        "y": ("mesh2d_nFaces", mesh.face_xy[:, 1]),
    }
)

print("Extracting water depths")
wd = extract_parameter("wd", MAX_STEP)
print("Extracting x velocities")
Vx = extract_parameter("Vx", MAX_STEP, shape=wd.shape[1:])
print("Extracting y velocities")
Vy = extract_parameter("Vy", MAX_STEP, shape=wd.shape[1:])

Extracting water depths
Extracting x velocities
Extracting y velocities


In [8]:
# The +1 on variables which are indicies is expected by graph_creation on loading in the NetCDF file.

simulation_output = xr.Dataset({
    "mesh2d_node_x": xr.DataArray(mesh.node_x, dims=["mesh2d_nNodes"]),
    "mesh2d_node_y": xr.DataArray(mesh.node_y, dims=["mesh2d_nNodes"]),
    "mesh2d_face_x": xr.DataArray(mesh.face_x, dims=["mesh2d_nFaces"]),
    "mesh2d_face_y": xr.DataArray(mesh.face_y, dims=["mesh2d_nFaces"]),
    "mesh2d_edge_nodes": xr.DataArray(mesh.edge_index.T + 1, dims=["mesh2d_nEdges", "Two"]),
    
    "mesh2d_edge_type": xr.DataArray(mesh.edge_type, dims=["mesh2d_nEdges"]),
    "mesh2d_edge_faces": xr.DataArray(edge_faces + 1, dims=["mesh2d_nEdges", "Two"]),
    "mesh2d_face_nodes": xr.DataArray(graph_creation.get_face_nodes_mesh(mesh) + 1, dims=["mesh2d_face_nodes", "mesh2d_nMax_face_nodes"]),

    "mesh2d_waterdepth": wd.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
    "mesh2d_ucx": Vx.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
    "mesh2d_ucy": Vy.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),

    "mesh2d_dem": xr.DataArray(mesh.DEM, dims=["mesh2d_nFaces"]),
})

In [10]:
simulation_output.reset_index("mesh2d_nFaces").to_netcdf("dyce_0.nc", format="NETCDF4")